# Домашнее задание: Binary Search on Answer на реальных данных

### Что нужно сделать

1. Найти **открытый датасет минимум на 100 000 строк** (будет супер, если какую-то сводку с помощью pandas (и мб numpy) ну и вашего текста вы напишете, чтобы я немного окунулась в сам датасет)
2. Разобраться, что означают его числовые признаки и какие особенности есть в данных
3. На **одном и том же датасете** придумать **минимум 3 разные постановки**, которые можно решать с помощью Binary Search on Answer
4. Для каждой постановки описать:
   - что требуется найти
   - что является `X`
   - что проверяет `check(X)`
   - направление монотонности и почему
   - границы `L` и `R`
5. Каждую из постановок полностью реализовать (будет поощряться, если вы потестите разные по сложности постановки, чтобы больше погрузиться в механику)
6. Для каждой реализованной задачи проверить, что найдено именно граничное значение
7. Для одной задачи провести эксперимент по времени работы на данных разного размера
8. Сделать короткие выводы

> **Важно:** три задачи должны отличаться не только числами. Нужны разные интерпретации одних и тех же данных

> Если в выбранных признаках есть отрицательные значения, нули, пропуски, выбросы или другие особенности - **не нужно автоматически их удалять**. Решите сами, как с ними работать, и обоснуйте это (мне без разницы, но как дата инженеры вы должны обдумать, как поступить лучше всего)

> Если `X` вещественный, укажите точность поиска `ε` и объясните, как вы останавливаете бинарный поиск. Если можно сформулировать задачу с целочисленным `X`, это обычно проще



## 0. Информация о работе

**ФИО:**  
Евграфов Богдан Владимирович

**Группа:**  
11-511

**Ссылка на датасет:**  
https://data.cityofchicago.org/d/ajtu-isnz

**Источник датасета:**  
Портал открытых данных города Чикаго, Taxi Trips 

**Кратко: что содержится в датасете?**  
Поездки такси в Чикаго. Одна строка - одна поездка: время посадки и высадки, длительность, пробег, районы посадки и высадки, тариф, чаевые, доплаты, тип оплаты, компания. Всего в датасете около 18 млн строк, я беру выборку 200 000 поездок (с 1 января 2025).

# 1. Загрузка датасета

Используйте любой удобный способ:

- загрузка по прямой ссылке;
- загрузка файла в Colab;
- Google Drive;
- API;
- другой способ.

Ниже оставлены несколько заготовок. Можно удалить лишние.

In [ ]:
import pandas as pd
import numpy as np
import math
import time

### Вариант A - загрузка CSV по ссылке

In [ ]:
DATASET_URL = "https://data.cityofchicago.org/resource/ajtu-isnz.csv?$limit=200000&$where=trip_start_timestamp%3E%3D%272025-01-01T00:00:00%27"

df = pd.read_csv(DATASET_URL)

### Проверка размера датасета

In [ ]:
print("Размер датасета:", df.shape)

if len(df) < 100_000:
    print("ВНИМАНИЕ: в датасете меньше 100 000 строк. Анна Дмитриевна расстроится((((")
else:
    print("Требование по размеру выполнено. Анна Дмитриевна довольна")

# 2. Быстрый анализ данных

Покажите, что вы понимаете, с какими данными работаете.

Нужно:

- посмотреть первые строки;
- изучить типы столбцов;
- выбрать один или несколько числовых признаков;
- проверить пропуски;
- заметить возможные отрицательные значения, нули, выбросы и другие особенности.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols

In [ ]:
df[numeric_cols].describe().T

In [ ]:
df[numeric_cols].isna().sum().sort_values(ascending=False).head(20)

In [ ]:
df["start"] = pd.to_datetime(df["trip_start_timestamp"])

print("период:", df["start"].min(), "-", df["start"].max())
print("fare < 0:", (df["fare"] < 0).sum())
print("fare == 0:", (df["fare"] == 0).sum())
print("trip_miles == 0:", (df["trip_miles"] == 0).sum())
print("trip_miles > 200:", (df["trip_miles"] > 200).sum())
print("trip_seconds == 0:", (df["trip_seconds"] == 0).sum())
print("пропуски fare / trip_miles / trip_seconds:", df["fare"].isna().sum(), df["trip_miles"].isna().sum(), df["trip_seconds"].isna().sum())

df[["fare", "trip_miles", "trip_seconds"]].quantile([0.01, 0.5, 0.99, 0.999, 1])

### Какие признаки вы выбрали?

**Признак(и):**  
fare, trip_miles, trip_seconds, trip_start_timestamp

**Что они означают?**  
fare - тариф поездки в долларах (без чаевых и доплат). trip_miles - пробег в милях. trip_seconds - длительность поездки в секундах. trip_start_timestamp - время начала поездки (в датасете округлено до 15 минут).

**Какие особенности данных обнаружены?**  
Время начала поездки округлено до 15 минут, поэтому у многих поездок оно совпадает. Есть пропуски в census_tract и в районах посадки и высадки (поездки за пределами города), но я эти столбцы не использую. Нулевые и очень большие значения тарифа, пробега и длительности смотрю по ячейке выше (количество нулей, максимумы и квантили).

**Как вы решили с ними работать и почему?**  
Строки из датафрейма я не удаляю, решаю отдельно для каждой задачи.  
В задаче 1 отрицательные и нулевые тарифы (если они есть) оставляю: в формуле потерь max(fare - C, 0) они дают 0 и не мешают. Выбросы тоже оставляю, потолок как раз нужен, чтобы их ограничивать.  
В задаче 2 каждую поездку нужно обработать, поэтому не удаляю. Пропуски и отрицательные пробеги заменяю на 0, а пробеги больше 200 миль обрезаю до 200, потому что для такси в городе это ошибка счётчика.  
В задаче 3 нужно сортировать длительности, поэтому поездки с пропущенной длительностью исключаю только из этой задачи.

# 3. Постановка №1

### 3.1. Что требуется найти?

Найти минимальный потолок цены поездки C (в центах): если все тарифы выше C урезать до C, то потери выручки должны быть не больше 1% от общей выручки.

### 3.2. Что является `X`?

`X = C, потолок тарифа в центах (целое число)`

### 3.3. Что проверяет `check(X)`?

`check(X) = сумма max(fare_i - C, 0) <= 1% от суммы положительных тарифов`

### 3.4. Направление монотонности

`FFFF | TTTT`

**Почему?**  
Чем больше C, тем меньше потери. Если C подходит, то C+1 тоже подходит. Ищем первый True.

### 3.5. Границы поиска

`L = 0`

`R = максимальный тариф в центах`

**Почему ответ гарантированно находится между ними?**  
При C = 0 теряется вся выручка, это больше 1%, значит check(0) = False. При C = максимальному тарифу ничего не урезается, потери 0, значит check(R) = True.

### 3.6. Почему эта постановка имеет смысл для выбранного датасета?

У тарифов длинный хвост из дорогих поездок, поэтому вопрос «какой потолок можно поставить, потеряв не больше 1%» имеет смысл.

# 4. Постановка №2

### 4.1. Что требуется найти?

Поездки, упорядоченные по времени посадки, нужно разрезать на K = 24 подряд идущих батча (по одному на воркер) так, чтобы самая большая нагрузка батча была как можно меньше. Нагрузка поездки - её пробег.

### 4.2. Что является `X`?

`X = максимально допустимая нагрузка одного батча (сумма пробегов, в сотых долях мили)`

### 4.3. Что проверяет `check(X)`?

`check(X) = можно ли разрезать поток на не более чем K батчей, в каждом из которых сумма пробегов <= X (проверяем жадно: набираем батч, пока не превысим X)`

### 4.4. Направление монотонности

`FFFF | TTTT`

**Почему?**  
Чем больше X, тем меньше батчей нужно. Если при X хватает K батчей, то при X+1 тоже хватает. Ищем первый True.

### 4.5. Границы поиска

`L = максимальный пробег одной поездки`

`R = сумма всех пробегов`

**Почему ответ гарантированно находится между ними?**  
Если X меньше самой длинной поездки, она не поместится ни в один батч. Если X равен сумме всех пробегов, всё помещается в один батч.

### 4.6. Почему эта постановка имеет смысл для выбранного датасета?

Это задача про разбиение потока на партиции с равной нагрузкой при сохранении порядка. Поездки сильно отличаются по длине, поэтому делить поровну по числу строк неправильно.

# 5. Постановка №3

### 5.1. Что требуется найти?

Для ручной проверки нужно выбрать K = 100 поездок так, чтобы длительности соседних выбранных поездок (по возрастанию) различались как можно сильнее, то есть выборка покрывала все типы поездок от коротких до длинных. Найти максимальный минимальный разрыв между длительностями.

### 5.2. Что является `X`?

`X = минимальный разрыв между длительностями соседних выбранных поездок, в секундах`

### 5.3. Что проверяет `check(X)`?

`check(X) = можно ли выбрать K поездок, у которых длительности отличаются от предыдущей выбранной не меньше чем на X (жадно: берём самую короткую, потом первую, у которой длительность >= предыдущая + X)`

### 5.4. Направление монотонности

`TTTT | FFFF`

**Почему?**  
Чем больше X, тем сложнее набрать K поездок. Если X подходит, то X-1 тоже подходит. Ищем последний True.

### 5.5. Границы поиска

`L = 0`

`R = (максимальная длительность - минимальная длительность) // (K - 1)`

**Почему ответ гарантированно находится между ними?**  
При X = 0 подойдут любые K поездок. K точек с разрывами не меньше X занимают не меньше (K-1)*X секунд, а всего у нас есть только максимальная - минимальная длительность.

### 5.6. Почему эта постановка имеет смысл для выбранного датасета?

Это выборка для ручной проверки качества данных, которая охватывает весь диапазон длительностей, а не только типичные короткие поездки. Из трёх задач эта самая придуманная.

## Проверка разнообразия постановок

**Чем задача №1 отличается от задачи №2?**  
В №1 считаем потери денег по всем тарифам, порядок строк не важен, X - сумма в центах. В №2 порядок строк важен, мы режем последовательность на части, X - вместимость батча.

**Чем задача №2 отличается от задачи №3?**  
В №2 нужно разложить все поездки по батчам и минимизировать максимум. В №3 мы выбираем только часть поездок и максимизируем минимальный разрыв, X - разница длительностей в секундах.

**Есть ли среди ваших задач разные направления монотонности (`FFFF|TTTT` и `TTTT|FFFF`)?**  
Да. №1 и №2 - FFFF|TTTT, №3 - TTTT|FFFF. По данным о поездках более естественны задачи вида «найти минимальный порог, чтобы уложиться в лимит», поэтому таких две.

# 6. Реализация первой выбранной задачи

Задача №1 (потолок тарифа).

- `X = C`, потолок тарифа в центах
- `check(X) = сумма max(fare_i - C, 0) <= 1% от суммы положительных тарифов`
- монотонность = FFFF | TTTT
- `L = 0`
- `R = максимальный тариф в центах`

In [ ]:
data = np.round(df["fare"].dropna().to_numpy() * 100).astype(np.int64)

LOSS_PERCENT = 1
print(len(data), data.min(), data.max())

In [ ]:
def check_1(x, data):
    gross = data[data > 0].sum()              # общая выручка (только положительные тарифы)
    loss = np.maximum(data - x, 0).sum()      # потери, если урезать тарифы до x
    return loss * 100 <= LOSS_PERCENT * gross

In [ ]:
def solve_1(data):
    left = 0
    right = int(data.max())

    while left < right:
        mid = (left + right) // 2
        if check_1(mid, data):
            right = mid
        else:
            left = mid + 1

    answer = left
    return answer

In [ ]:
answer_1 = solve_1(data)
print("Ответ:", answer_1, "центов =", answer_1 / 100, "долларов")

### Проверка границы

Если вы ищете **первый `True`**, должно выполняться:

```text
check(answer) = True
check(answer - 1) = False
```

Если вы ищете **последний `True`**:

```text
check(answer) = True
check(answer + 1) = False
```

Если соседнее значение выходит за допустимый диапазон, объясните это

In [ ]:
print("check(answer) =", check_1(answer_1, data))
print("check(answer - 1) =", check_1(answer_1 - 1, data))

### Сложность решения №1

**Сложность `check(X)`:**  
O(N), один проход по данным.

**Количество итераций бинарного поиска:**  
около log2(R - L), то есть примерно 20-30.

**Общая сложность:**  
O(N * log(R - L)).

**Дополнительная память:**  
O(N), numpy создаёт временный массив data - x.

# 7. Реализация второй выбранной задачи

Задача №2 (батчи).

- `X` = максимальная нагрузка одного батча (сотые доли мили)
- `check(X)` = можно ли разрезать поток на не более чем K = 24 батча с суммой <= X
- монотонность = FFFF | TTTT
- `L = максимальный пробег одной поездки`
- `R = сумма всех пробегов`

In [ ]:
K_BATCHES = 24
MAX_MILES = 200

order = np.argsort(df["start"].to_numpy(), kind="stable")
miles = df["trip_miles"].to_numpy()[order]

miles = np.clip(np.nan_to_num(miles, nan=0.0), 0, MAX_MILES)
weights = np.round(miles * 100).astype(np.int64)

data_2 = np.concatenate(([0], np.cumsum(weights)))
print(len(weights), data_2[-1])

In [ ]:
def check_2(x, data):
    P = data
    n = len(P) - 1
    pos = 0
    batches = 0

    while pos < n:
        if batches == K_BATCHES:
            return False

        j = int(np.searchsorted(P, P[pos] + x, side="right")) - 1
        if j == pos:          
            return False
        pos = j
        batches += 1

    return True

In [ ]:
def solve_2(data):
    left = int(np.diff(data).max())
    right = int(data[-1])

    while left < right:
        mid = (left + right) // 2
        if check_2(mid, data):
            right = mid
        else:
            left = mid + 1

    answer = left
    return answer

In [ ]:
answer_2 = solve_2(data_2)
print("Ответ:", answer_2, "сотых миль =", answer_2 / 100, "миль")

In [ ]:
print("check(answer) =", check_2(answer_2, data_2))
print("check(answer - 1) =", check_2(answer_2 - 1, data_2))

### Сложность решения №2

**Сложность `check(X)`:**  
O(K * log N): не больше K шагов, каждый шаг - бинарный поиск по префиксным суммам.

**Количество итераций бинарного поиска:**  
около log2(R - L), примерно 25-30.

**Общая сложность:**  
O(N) на подготовку (сортировка O(N log N), префиксные суммы) и O(K * log N * log(R - L)) на сам поиск.

**Дополнительная память:**  
O(N) на массив префиксных сумм.

# 7.1. Третья задача (дополнительно)

Задача №3 (выборка по длительности).

- `X` = минимальный разрыв между длительностями соседних выбранных поездок (секунды)
- `check(X)` = можно ли выбрать K = 100 поездок с разрывами >= X
- монотонность = TTTT | FFFF
- `L = 0`
- `R = (максимальная длительность - минимальная длительность) // (K - 1)`

In [ ]:
K_SAMPLE = 100

dur = df["trip_seconds"].dropna()
print("исключено из этой задачи (нет длительности):", len(df) - len(dur))

data_3 = np.sort(dur.to_numpy().astype(np.int64))
print(len(data_3), data_3[0], data_3[-1])

In [ ]:
def check_3(x, data):
    n = len(data)
    if x == 0:
        return n >= K_SAMPLE

    count = 1
    last = int(data[0])

    while count < K_SAMPLE:
        idx = int(np.searchsorted(data, last + x, side="left"))
        if idx >= n:
            return False
        last = int(data[idx])
        count += 1

    return True

In [ ]:
def solve_3(data):
    left = 0
    right = int((data[-1] - data[0]) // (K_SAMPLE - 1))

    while left < right:
        mid = (left + right + 1) // 2    
        if check_3(mid, data):
            left = mid
        else:
            right = mid - 1

    answer = left
    return answer

In [ ]:
answer_3 = solve_3(data_3)
print("Ответ:", answer_3, "секунд")

print("check(answer) =", check_3(answer_3, data_3))
print("check(answer + 1) =", check_3(answer_3 + 1, data_3))

# 8. Эксперимент на больших данных

Для **одной** из двух реализованных задач измерьте время работы на:

- 1 000 строк;
- 10 000 строк;
- 100 000 строк;
- полном датасете.

Выберите ту реализацию, которую хотите исследовать.

> Если ваша постановка зависит от нескольких столбцов, формируйте подвыборку согласованно.

In [ ]:
# Укажите функцию решения, которую будете измерять:
solve_fn = solve_1

# Укажите данные:
full_data = data

sizes = [1_000, 10_000, 100_000, len(full_data)]
sizes = sorted(set(n for n in sizes if n <= len(full_data)))

results = []

for n in sizes:
    subset = full_data[:n]

    start = time.perf_counter()
    answer = solve_fn(subset)
    elapsed = time.perf_counter() - start

    results.append({
        "N": n,
        "answer": answer,
        "time_sec": elapsed
    })

timing_df = pd.DataFrame(results)
timing_df

### Что получилось?

**Как меняется время работы при увеличении `N`?**  
Время растёт примерно пропорционально N (в 10 раз больше строк - примерно в 10 раз больше времени), число итераций почти не меняется.

**Соответствует ли это вашей теоретической оценке сложности?**  
Да, O(N * log(R - L)): каждый check проходит по всем данным, а итераций одно и то же количество.

**Что сильнее влияет на время: размер датасета или ширина диапазона `[L, R]`? Почему?**  
Сильнее влияет размер датасета. Он умножает время на каждой проверке, а ширина диапазона влияет только через логарифм: если R станет больше в 1000 раз, добавится около 10 итераций.

# 9. Сравнение с перебором

**Сколько примерно различных значений `X` пришлось бы проверить перебором?**  
Если перебирать C по одному центу снизу вверх, то примерно ответ + 1 значений (в худшем случае R + 1). См. оценку ниже.

**Сколько проверок делает Binary Search?**  
Около 20-30.

**Почему Binary Search on Answer здесь выигрывает?**  
Каждая проверка отбрасывает половину оставшихся значений, а не одно значение.

In [ ]:
start = time.perf_counter()
check_1(answer_1, data)
t_check = time.perf_counter() - start

print("время одного check:", t_check, "сек")
print("перебор до ответа:", answer_1 + 1, "проверок, примерно", (answer_1 + 1) * t_check / 60, "минут")
print("бинарный поиск: около", math.ceil(math.log2(data.max() + 1)), "проверок")

In [ ]:
def brute_force_1(data):
    for c in range(int(data.max()) + 1):
        if check_1(c, data):
            return c

small = data[:300_000]
small = small[small <= 30_000][:20_000]

start = time.perf_counter()
bf = brute_force_1(small)
t_bf = time.perf_counter() - start

start = time.perf_counter()
bs = solve_1(small)
t_bs = time.perf_counter() - start

print("перебор:", bf, t_bf, "сек")
print("бинарный поиск:", bs, t_bs, "сек")
print("ответы совпали:", bf == bs)

# 10. Выводы

### 1. Какая из трёх постановок оказалась самой естественной для этого датасета? Почему?

Первая (потолок тарифа). Тарифы есть в таблице напрямую, а вопрос про потолок похож на реальный. Странные значения (отрицательные, нулевые) не пришлось чистить, они не мешают формуле.

### 2. Какая постановка получилась наиболее искусственной? Почему?

Третья (выборка по длительности). Число K = 100 выбрано произвольно, и смысл выбирать поездки так не очень понятен. Я придумал её, чтобы получить другое направление монотонности.

### 3. Что в Binary Search on Answer оказалось самым сложным?

- придумать `X`;
- придумать `check(X)`;
- доказать монотонность;
- подобрать `L/R`;
- написать бинарный поиск;
- другое.

_Ваш ответ: придумать check(X) и понять, что жадная проверка правильная. Ещё легко ошибиться на единицу при округлении mid и в границах L/R (для последнего True нужен mid = (left + right + 1) // 2)._

### 4. В каких случаях Binary Search on Answer здесь был действительно полезен?

Когда диапазон ответа очень большой, а проверить одно значение легко. В первой задаче диапазон - миллионы центов, а бинарный поиск делает около 20-30 проверок. Во второй и третьей задачах перебрать все разбиения или все подмножества вообще невозможно, а с бинарным поиском и жадной проверкой задача решается быстро.